In [34]:
import os, sys

PROJECT_ROOT = r"C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from model import ConvNeXtV2TinyScratch

In [35]:
import os, json
import numpy as np
import pandas as pd

OUT_DIR = r"C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\ConvUnet\Baseline+ECA\Baseline+ECA(SamplerUNet++)GEMopt_Output"
KB_IDS = set()

need = [
    "checkpoint_infer.pt",
    "checkpoint_infer.json",
    "embeddings.npy",
    "rag_meta.csv",
]
for f in need:
    p = os.path.join(OUT_DIR, f)
    print(f, "OK" if os.path.exists(p) else "MISSING", p)

checkpoint_infer.pt OK C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\ConvUnet\Baseline+ECA\Baseline+ECA(SamplerUNet++)GEMopt_Output\checkpoint_infer.pt
checkpoint_infer.json OK C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\ConvUnet\Baseline+ECA\Baseline+ECA(SamplerUNet++)GEMopt_Output\checkpoint_infer.json
embeddings.npy OK C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\ConvUnet\Baseline+ECA\Baseline+ECA(SamplerUNet++)GEMopt_Output\embeddings.npy
rag_meta.csv OK C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\ConvUnet\Baseline+ECA\Baseline+ECA(SamplerUNet++)GEMopt_Output\rag_meta.csv


In [36]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd

# =========================
# 0) Load RAG library assets
# =========================
meta = pd.read_csv(os.path.join(OUT_DIR, "rag_meta.csv")).reset_index(drop=True)
embs = np.load(os.path.join(OUT_DIR, "embeddings.npy")).astype("float32")
assert len(meta) == embs.shape[0], f"meta rows={len(meta)} != embs rows={embs.shape[0]}"

def guess_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

PATH_COL   = guess_col(meta, ["full_path","image_path","img_path","path","filepath","file_path","png_path"])
CASEID_COL = guess_col(meta, ["image_id","ImageId","case_id","id","new_filename","filename","file_name"])
SPLIT_COL  = guess_col(meta, ["split","subset","set","stage","partition"])
YTRUE_COL  = guess_col(meta, ["y_true","label","gt","target","truth"]) 

print("PATH_COL:", PATH_COL)
print("CASEID_COL:", CASEID_COL)
print("SPLIT_COL:", SPLIT_COL)
print("YTRUE_COL:", YTRUE_COL)

if PATH_COL is None:
    raise ValueError(f"Can't find a path column in metadata. columns={list(meta.columns)}")

cfg_path = os.path.join(OUT_DIR, "checkpoint_infer.json")
cfg = json.load(open(cfg_path, "r", encoding="utf-8")) if os.path.exists(cfg_path) else {}
THR = float(cfg.get("threshold", cfg.get("thr", 0.5)))
T   = float(cfg.get("temperature_T", cfg.get("T", 1.0)))
print("THR:", THR, "T:", T)

PCOL = guess_col(meta, ["p_calibrated","p","prob","probability","pred_prob"])
YPRED_COL = guess_col(meta, ["y_pred","pred","yhat","prediction"])

_path2row = {str(p): i for i, p in enumerate(meta[PATH_COL].astype(str).tolist())}

USE_LOOKUP_ONLY = False


import torch
from PIL import Image
import torchvision.transforms as tvT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- 读取 checkpoint（优先从 pt 里取 preprocess/postprocess/model_kwargs）----
ckpt_path = os.path.join(OUT_DIR, "checkpoint_infer.pt")
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"checkpoint not found: {ckpt_path}")

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

# ---- postprocess: threshold + temperature ----
post = ckpt.get("postprocess", {}) or cfg.get("postprocess", {}) or cfg
THR = float(post.get("cls_threshold", post.get("threshold", post.get("thr", THR))))
T   = float(post.get("temperature_T", post.get("T", T)))
print("THR:", THR, "T:", T)

# ---- preprocess: img_size/mean/std ----
pre = ckpt.get("preprocess", {}) or cfg.get("preprocess", {}) or cfg

def _as_list(v, default):
    # 兼容 v=0.5 / v=[0.5] / v=np.array([0.5])
    if v is None:
        return [float(default)]
    if isinstance(v, (list, tuple)):
        return [float(x) for x in v]
    try:
        import numpy as _np
        if isinstance(v, _np.ndarray):
            return [float(x) for x in v.tolist()]
    except Exception:
        pass
    return [float(v)]

IMG_SIZE = int(pre.get("img_size", pre.get("image_size", cfg.get("img_size", cfg.get("image_size", 512)))))
MEAN = _as_list(pre.get("mean", pre.get("img_mean", cfg.get("mean", cfg.get("img_mean", 0.0)))), default=0.0)
STD  = _as_list(pre.get("std",  pre.get("img_std",  cfg.get("std",  cfg.get("img_std",  1.0)))), default=1.0)

# 单通道：如果 mean/std 给了多通道，只取第一个
MEAN = [MEAN[0]]
STD  = [STD[0]]

preprocess = tvT.Compose([
    tvT.Grayscale(num_output_channels=1),
    tvT.Resize((IMG_SIZE, IMG_SIZE)),
    tvT.ToTensor(),
    tvT.Normalize(mean=MEAN, std=STD),
])

print("IMG_SIZE:", IMG_SIZE, "MEAN:", MEAN, "STD:", STD)

# ---- build model from your model.py (你在文件顶部已经: from model import ConvNeXtV2TinyScratch) ----
model_kwargs = ckpt.get("model_kwargs", {}) or cfg.get("model_kwargs", {})
if not model_kwargs:
    # 兜底：按你训练时常用的参数
    model_kwargs = dict(in_chans=1, n_classes=1, drop_path_rate=0.1, use_seg_guided=False)

print("model_kwargs:", model_kwargs)

infer_model = ConvNeXtV2TinyScratch(**model_kwargs).to(device)

# ---- load state dict（兼容不同键名 + module. 前缀）----
state = (
    ckpt.get("model_state")
    or ckpt.get("model_state_dict")
    or ckpt.get("state_dict")
    or ckpt.get("model")
)
if state is None:
    raise ValueError(f"Checkpoint keys={list(ckpt.keys())}, but no state_dict found.")

if any(k.startswith("module.") for k in state.keys()):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}

miss, unexp = infer_model.load_state_dict(state, strict=False)
if miss:
    print("[Warn] missing keys (show first 10):", miss[:10])
if unexp:
    print("[Warn] unexpected keys (show first 10):", unexp[:10])

infer_model.eval()

import time
import torch.nn.functional as F
from PIL import ImageDraw

def save_seg_overlay_png(
    image_path: str,
    seg_logits: torch.Tensor,
    out_png: str,
    thr: float = 0.5,
    alpha: float = 0.35,
    draw_bbox: bool = True,
):
    """
    Saves a visualization PNG with segmentation overlay and optional bounding box.
    """
    base = Image.open(image_path).convert("RGB")
    w, h = base.size

    prob = torch.sigmoid(seg_logits)  # [1,1,H,W]
    prob = F.interpolate(prob, size=(h, w), mode="bilinear", align_corners=False)[0, 0]
    prob_np = prob.detach().cpu().numpy()  # [h,w], 0~1

    red = (prob_np * 255).clip(0, 255).astype(np.uint8)
    overlay_np = np.zeros((h, w, 3), dtype=np.uint8)
    overlay_np[..., 0] = red  # R通道
    overlay_img = Image.fromarray(overlay_np, mode="RGB")

    blended = Image.blend(base, overlay_img, float(alpha))

    if draw_bbox:
        mask = prob_np >= float(thr)
        if mask.any():
            ys, xs = np.where(mask)
            x1, x2 = int(xs.min()), int(xs.max())
            y1, y2 = int(ys.min()), int(ys.max())
            draw = ImageDraw.Draw(blended)
            lw = max(2, int(min(w, h) * 0.005))
            draw.rectangle([x1, y1, x2, y2], outline=(255, 255, 0), width=lw)

    os.makedirs(os.path.dirname(out_png), exist_ok=True)
    blended.save(out_png)
    return out_png

@torch.no_grad()
def infer_one(image_path: str):
    img = Image.open(image_path).convert("L")
    x = preprocess(img).unsqueeze(0).to(device)

    embed_buf = []
    def _hook(_m, _inp, out):
        embed_buf.append(out.detach())

    handle = infer_model.backbone.norm_head.register_forward_hook(_hook)

    cls_logits, seg_logits = infer_model(x)  # forward return (cls_logits, seg_logits)
    handle.remove()

    if len(embed_buf) == 0:
        raise RuntimeError("Hook did not capture embedding from infer_model.backbone.norm_head")

    emb = embed_buf[0]  
    logits = cls_logits.view(-1).float()

    p = torch.sigmoid(logits / float(T)).item()
    yhat = int(p >= float(THR))

    emb_np = emb.detach().cpu().numpy().astype("float32")
    if emb_np.ndim == 1:
        emb_np = emb_np[None, :]

    # Make sure the embedding dimension matches the library embeddings
    if emb_np.shape[1] != embs.shape[1]:
        raise RuntimeError(f"Embedding dim mismatch: query D={emb_np.shape[1]} vs library D={embs.shape[1]}")

    # Generate and save segmentation overlay PNG for visualization
    base_report_dir = globals().get("REPORT_DIR", os.path.join(OUT_DIR, "reports"))
    overlay_dir = os.path.join(base_report_dir, "overlays")

    stem = Path(image_path).stem
    ts = time.strftime("%Y%m%d_%H%M%S")
    overlay_path = os.path.join(overlay_dir, f"{stem}_{ts}_overlay.png")

    save_seg_overlay_png(
        image_path=image_path,
        seg_logits=seg_logits,
        out_png=overlay_path,
        thr=0.5,       
        alpha=0.35,
        draw_bbox=True
    )
    return emb_np, float(p), int(yhat), overlay_path

# =========================
# 3) Unified interface used by RAG
# =========================
def embed_and_predict(image_path: str):
    if USE_LOOKUP_ONLY:
        # ---- demo: only for images in meta ----
        key = str(image_path)
        if key not in _path2row:
            raise KeyError(f"image_path not found in meta[{PATH_COL}]: {key}")

        i = _path2row[key]
        emb = embs[i:i+1]  # shape (1, D)

        row = meta.iloc[i]
        p = float(row[PCOL]) if (PCOL is not None and PCOL in meta.columns) else 0.5
        yhat = int(row[YPRED_COL]) if (YPRED_COL is not None and YPRED_COL in meta.columns) else int(p >= THR)
        return emb, float(p), int(yhat), None

    # ---- end-to-end: for any new image ----
    return infer_one(image_path)

PATH_COL: full_path
CASEID_COL: image_id
SPLIT_COL: None
YTRUE_COL: y_true
THR: 0.5 T: 0.8587129712104797
THR: 0.37 T: 0.8587129712104797
IMG_SIZE: 576 MEAN: [0.5] STD: [0.25]
model_kwargs: {'in_chans': 1, 'n_classes': 1, 'drop_path_rate': 0.1, 'use_seg_guided': False}


In [37]:
import numpy as np
import faiss

def _l2norm(v: np.ndarray) -> np.ndarray:
    v = v.astype("float32")
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-12)

embs_norm = _l2norm(embs)

N = len(meta)

if SPLIT_COL is None:
    print("[Warn] SPLIT_COL is None -> use full meta as retrieval pool.")
    train_rowids = np.arange(N, dtype=np.int64)
else:
    split = meta[SPLIT_COL].astype(str).str.lower()
    train_rowids = np.flatnonzero(split.isin(["train", "tr"])).astype(np.int64)
    if train_rowids.size == 0:
        print("[Warn] No train rows found in SPLIT_COL -> use full meta as retrieval pool.")
        train_rowids = np.arange(N, dtype=np.int64)

if train_rowids.size == 0:
    print("[Warn] train_rowids is empty -> use full meta as retrieval pool.")
    train_rowids = np.arange(N, dtype=np.int64)

train_embs = embs_norm[train_rowids]
train_index = faiss.IndexFlatIP(train_embs.shape[1])
train_index.add(train_embs)

print("train_index size:", train_index.ntotal, "dim:", train_embs.shape[1])

def image_retrieve_by_vector(q_vec: np.ndarray, topk: int = 5, exclude_case_id=None):
    q = _l2norm(q_vec)
    extra = 10
    scores, idxs = train_index.search(q, topk + extra)

    row_ids = train_rowids[idxs[0]].astype(int)
    hits = meta.iloc[row_ids].copy()
    hits["sim"] = scores[0]

    if exclude_case_id is not None:
        if CASEID_COL is not None and CASEID_COL in hits.columns:
            hits = hits[hits[CASEID_COL].astype(str) != str(exclude_case_id)]
        else:
            from pathlib import Path
            hits = hits[hits[PATH_COL].astype(str).apply(lambda p: Path(p).name) != str(exclude_case_id)]

    return hits.head(topk)

[Warn] SPLIT_COL is None -> use full meta as retrieval pool.
train_index size: 12047 dim: 768


In [38]:
from pathlib import Path

def get_case_id_from_path(image_path: str) -> str:
    return Path(str(image_path)).name 

def shrink_hits_for_llm(hits_df: pd.DataFrame, topk: int):
    out = []
    for _, row in hits_df.head(topk).iterrows():
        item = {
            "case_id": str(row[CASEID_COL]) if (CASEID_COL is not None and CASEID_COL in hits_df.columns)
                      else get_case_id_from_path(row[PATH_COL]),
            "sim": float(row["sim"]) if "sim" in hits_df.columns else None
        }
        for c in ["p_calibrated","p","prob","y_pred","pred","logit"]:
            if c in hits_df.columns:
                v = row[c]
                try:
                    item[c] = int(v) if c in ["y_pred","pred"] else float(v)
                except Exception:
                    item[c] = str(v)
        out.append(item)
    return out

In [39]:
def image_rag_for_image(image_path: str, topk: int = 5):
    emb, p, yhat, overlay_path = embed_and_predict(image_path)

    # exclude self ONLY if the query image exists in the retrieval library
    exclude_id = None
    key = str(image_path)
    if key in _path2row:
        if CASEID_COL is not None and CASEID_COL in meta.columns:
            m = meta[meta[PATH_COL].astype(str) == key]
            exclude_id = str(m.iloc[0][CASEID_COL]) if len(m) > 0 else None
        else:
            exclude_id = None

    hits = image_retrieve_by_vector(emb, topk=topk, exclude_case_id=exclude_id)

    pos_rate_true = None
    if YTRUE_COL is not None and YTRUE_COL in hits.columns:
        try:
            pos_rate_true = float(hits[YTRUE_COL].mean())
        except Exception:
            pos_rate_true = None

    similar_public = shrink_hits_for_llm(hits, topk=topk)
    retrieved_case_ids = [x.get("case_id") for x in similar_public if isinstance(x, dict)]

    return {
        "p_calibrated": float(p),
        "y_pred": int(yhat),
        "threshold": float(THR),
        "temperature_T": float(T),

        "overlay_path": overlay_path,
        "overlay_filename": os.path.basename(overlay_path) if overlay_path else None,

        "similar_cases": similar_public,
        "retrieved_case_ids": retrieved_case_ids,

        "debug": {
            "exclude_case_id": exclude_id,
            "sim_mean": float(hits["sim"].mean()) if "sim" in hits.columns else None,
            "pos_rate_true": pos_rate_true,
        }
    }


In [40]:
import re

SAFE_NEGATION_PATTERNS = [
    r"cannot assess",
    r"cannot be assessed",
    r"cannot be determined",
    r"not available",
    r"no localization information",
    r"no laterality information",
    r"size assessment is not available",
    r"extent cannot be determined",
    r"tension physiology cannot be determined",
]

UNSUPPORTED_ASSERTION_PATTERNS = [
    r"\bleft pneumothorax\b",
    r"\bright pneumothorax\b",
    r"\bsmall pneumothorax\b",
    r"\blarge pneumothorax\b",
    r"\bmoderate pneumothorax\b",
    r"\btension pneumothorax\b",
    r"\bapical pneumothorax\b",
    r"\bbasal pneumothorax\b",
    r"\bvisible pleural line\b",
    r"\bpleural line is seen\b",
    r"\blung collapse\b",
    r"\bpartial collapse\b",
    r"\bpercentage collapse\b",
]

def normalize_text_for_safety(text: str) -> str:
    s = str(text or "").lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def unsupported_detail_flags(text: str) -> dict:
    s = normalize_text_for_safety(text)

    flags = {
        "unsupported_localization": False,
        "unsupported_size": False,
        "unsupported_tension": False,
        "unsupported_imaging_sign": False,
        "unsupported_other": False,
    }

    safe_localization_patterns = [
        r"localization is not available",
        r"no localization information",
        r"laterality is not available",
        r"laterality cannot be determined",
    ]

    safe_size_patterns = [
        r"size assessment is not available",
        r"size cannot be assessed",
        r"extent cannot be determined",
        r"extent is not available",
    ]

    safe_tension_patterns = [
        r"tension cannot be determined",
        r"tension physiology cannot be determined",
        r"tension is not assessable",
        r"no assessment of tension",
    ]

    bad_localization_patterns = [
        r"\bleft pneumothorax\b",
        r"\bright pneumothorax\b",
        r"\bapical pneumothorax\b",
        r"\bbasal pneumothorax\b",
        r"\bleft-sided\b",
        r"\bright-sided\b",
    ]

    bad_size_patterns = [
        r"\bsmall pneumothorax\b",
        r"\blarge pneumothorax\b",
        r"\bmoderate pneumothorax\b",
        r"\btrace pneumothorax\b",
        r"\bsize of the pneumothorax\b",
        r"\bextent of the pneumothorax\b",
    ]

    bad_tension_patterns = [
        r"\btension pneumothorax\b",
        r"\bfindings suggest tension\b",
    ]

    bad_sign_patterns = [
        r"\bvisible pleural line\b",
        r"\bpleural line is seen\b",
        r"\blung collapse\b",
        r"\bpartial collapse\b",
    ]

    if any(re.search(p, s) for p in bad_localization_patterns):
        if not any(re.search(p, s) for p in safe_localization_patterns):
            flags["unsupported_localization"] = True

    if any(re.search(p, s) for p in bad_size_patterns):
        if not any(re.search(p, s) for p in safe_size_patterns):
            flags["unsupported_size"] = True

    if any(re.search(p, s) for p in bad_tension_patterns):
        if not any(re.search(p, s) for p in safe_tension_patterns):
            flags["unsupported_tension"] = True

    if any(re.search(p, s) for p in bad_sign_patterns):
        flags["unsupported_imaging_sign"] = True

    flags["unsupported_other"] = any([
        flags["unsupported_localization"],
        flags["unsupported_size"],
        flags["unsupported_tension"],
        flags["unsupported_imaging_sign"],
    ])

    return flags

def hallucination_flag(report: dict) -> bool:
    text = str((report or {}).get("diagnostic_report", ""))
    flags = unsupported_detail_flags(text)
    return any(flags.values())

In [41]:
import re

def word_count_en(text: str) -> int:
    return len(re.findall(r"\b[\w']+\b", text or ""))

def soft_too_long(text: str, max_words: int = 230, max_chars: int = 1600) -> bool:
    return (word_count_en(text) > max_words) or (len(text) > max_chars)

In [42]:
def fail_safe_report(payload: dict, reason: str) -> dict:
    print("[FAIL_SAFE_REASON]", reason)

    msg = (
        "Clinical context: AI-assisted pneumothorax screening output could not be safely summarized. "
        "Technique: Image-based automated screening output only. "
        "Findings: A reliable concise diagnostic summary could not be generated from the current processing pathway. "
        "Impression: Indeterminate automated report state. "
        "Recommendations: Recommend radiologist review and clinical correlation. "
        "Limitations: This fallback report was issued because the automated reporting safeguard was triggered."
    )

    visual = (payload or {}).get("visual_support", {}) or {}

    return {
        "diagnostic_report": msg,
        "evidence": {
            "text_chunk_ids": [],
            "retrieved_case_ids": [],
        },
        "visual_support": {
            "overlay_available": bool(visual.get("overlay_available", False)),
            "overlay_path": visual.get("overlay_path"),
            "overlay_filename": visual.get("overlay_filename"),
            "overlay_note": visual.get("overlay_note", ""),
        },
        "fail_safe": True,
        "fail_reason": str(reason),
    }

In [43]:
import requests

API_KEY = "sk-6de58e90d64a47ad9163e14cc50067aa"

URL = "https://api.deepseek.com/chat/completions"
MODEL = "deepseek-chat"

def call_llm(system: str, user: str) -> str:
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "stream": False,
        "temperature": 0,
    }
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    r = requests.post(URL, json=payload, headers=headers, timeout=180)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

In [44]:
SYSTEM = """You are a consultant radiologist drafting a safe AI-assisted diagnostic report for pneumothorax screening.

Write one concise professional English report using these inline headings:
Clinical context, Technique, Findings, Impression, Recommendations, Limitations.

Use the inputs as follows:
- prediction.narrative_label and prediction.confidence_band determine the overall wording strength.
- text_rag.evidence_chunks may be used ONLY for safe wording, limitation statements, uncertainty wording, and generic recommendations.
- image_rag.behaviour_context may be used ONLY to make the wording more cautious or more stable; it MUST NOT be treated as direct clinical evidence.

Strict rules:
1. For positive outputs, prefer wording such as 'suggestive of pneumothorax'.
2. For negative outputs, prefer wording such as 'not suggestive of pneumothorax'.
3. For borderline confidence or conflicting/limited contextual support, prefer 'indeterminate' or clearly cautious wording.
4. Never infer laterality, localization, size, extent, pleural line, collapse, or tension physiology.
5. It is acceptable to state that localization, laterality, size, extent, or tension assessment is not available from this output.
6. Do NOT mention model, probability, threshold, calibration, retrieval, RAG, chunk, case ID, prompt, code, or JSON.
7. Do NOT mention attached images, overlays, filenames, or file paths in the diagnostic report text.
8. Keep the report concise, clinically styled, and faithful to the provided evidence only.

Output MUST be valid JSON with exactly these keys:
{
  "diagnostic_report": "...",
  "evidence": {
    "text_chunk_ids": [],
    "retrieved_case_ids": []
  }
}
"""

USER_TMPL = """Input JSON:
{payload}

Return ONLY JSON in this exact format:
{{
  "diagnostic_report": "...",
  "evidence": {{
    "text_chunk_ids": [],
    "retrieved_case_ids": []
  }}
}}
"""

In [45]:
import json

def extract_json_object(s: str) -> str:
    s = s.strip()
    l = s.find("{")
    r = s.rfind("}")
    if l == -1 or r == -1 or r <= l:
        raise ValueError("No JSON object found in LLM output.")
    return s[l:r+1]

def build_user_prompt(payload: dict) -> str:
    payload_json = json.dumps(payload, ensure_ascii=False, indent=2)
    return USER_TMPL.format(payload=payload_json)

def generate_report(payload: dict) -> dict:
    user = build_user_prompt(payload)
    raw = call_llm(SYSTEM, user)

    text = ""
    evidence = {
        "text_chunk_ids": [],
        "retrieved_case_ids": [],
    }

    try:
        obj = extract_json_object(raw)
        rep = json.loads(obj)

        if isinstance(rep, dict):
            text = str(rep.get("diagnostic_report", "")).strip()

            ev = rep.get("evidence", {}) or {}
            txt_ids = ev.get("text_chunk_ids", []) or []
            case_ids = ev.get("retrieved_case_ids", []) or []

            evidence = {
                "text_chunk_ids": list(map(str, txt_ids)) if isinstance(txt_ids, list) else [],
                "retrieved_case_ids": list(map(str, case_ids)) if isinstance(case_ids, list) else [],
            }

    except Exception:
        pass

    if not text:
        text = str(raw).strip()

    return {
        "diagnostic_report": text,
        "evidence": evidence,
    }

In [46]:
def evidence_ok(report: dict, payload: dict) -> bool:
    ev = report.get("evidence", {}) or {}

    txt_ids = ev.get("text_chunk_ids", []) or []
    case_ids = ev.get("retrieved_case_ids", []) or []

    # text ids must come from payload.text_rag if provided
    if "text_rag" in payload:
        allowed_txt = {c["chunk_id"] for c in payload["text_rag"].get("evidence_chunks", [])}
        if not (isinstance(txt_ids, list) and set(txt_ids).issubset(allowed_txt)):
            return False
    else:
        if txt_ids != []:
            return False

    # case ids must come from payload.image_rag if provided
    if "image_rag" in payload:
        allowed_case = set(map(str, payload["image_rag"].get("retrieved_case_ids", []) or []))
        if not (isinstance(case_ids, list) and set(map(str, case_ids)).issubset(allowed_case)):
            return False
    else:
        if case_ids != []:
            return False

    return True

In [47]:
def sanitize_report_structure(report: dict, payload: dict) -> dict:
    text = str((report or {}).get("diagnostic_report", "")).strip()

    ev = (report or {}).get("evidence", {}) or {}
    txt_ids = ev.get("text_chunk_ids", []) or []
    case_ids = ev.get("retrieved_case_ids", []) or []

    cleaned = {
        "diagnostic_report": text,
        "evidence": {
            "text_chunk_ids": list(map(str, txt_ids)) if isinstance(txt_ids, list) else [],
            "retrieved_case_ids": list(map(str, case_ids)) if isinstance(case_ids, list) else [],
        }
    }

    if not evidence_ok(cleaned, payload):
        cleaned["evidence"] = {
            "text_chunk_ids": [],
            "retrieved_case_ids": [],
        }

    return cleaned

In [48]:
def generate_report_safe(payload: dict) -> dict:
    try:
        rep = generate_report(payload)
    except Exception as e:
        return fail_safe_report(payload, f"exception during generation: {e}")

    rep = sanitize_report_structure(rep, payload)
    text = str(rep.get("diagnostic_report", "")).strip()

    if not text:
        return fail_safe_report(payload, "empty diagnostic_report")

    flags = unsupported_detail_flags(text)
    need_safety_rewrite = any(flags.values())
    need_length_rewrite = ("soft_too_long" in globals() and soft_too_long(text))

    if need_safety_rewrite or need_length_rewrite:
        rewrite_instruction = """
        Rewrite the report to be concise, professional, and safe.
        Do NOT invent localization, laterality, size, extent, tension physiology, pleural line, collapse, or other unsupported imaging details.
        You may explicitly say that these assessments are not available from the current output.
        Return JSON only in this exact format:
        {
        "diagnostic_report": "...",
        "evidence": {
            "text_chunk_ids": [],
            "retrieved_case_ids": []
        }
        }
        """

        fix_user = rewrite_instruction + "\n\nOriginal report:\n" + text

        try:
            raw2 = call_llm(SYSTEM, fix_user)
            obj2 = extract_json_object(raw2)
            rep2 = json.loads(obj2)

            if isinstance(rep2, dict):
                rep = {
                    "diagnostic_report": str(rep2.get("diagnostic_report", "")).strip(),
                    "evidence": rep2.get("evidence", {
                        "text_chunk_ids": [],
                        "retrieved_case_ids": [],
                    })
                }
                rep = sanitize_report_structure(rep, payload)

                text2 = str(rep.get("diagnostic_report", "")).strip()
                if not text2:
                    return fail_safe_report(payload, "rewrite returned empty diagnostic_report")

                flags2 = unsupported_detail_flags(text2)
                if any(flags2.values()):
                    return fail_safe_report(payload, f"unsafe after rewrite: {flags2}")

        except Exception as e:
            return fail_safe_report(payload, f"rewrite failed: {e}")

    visual = (payload or {}).get("visual_support", {}) or {}
    rep["visual_support"] = {
        "overlay_available": bool(visual.get("overlay_available", False)),
        "overlay_path": visual.get("overlay_path"),
        "overlay_filename": visual.get("overlay_filename"),
        "overlay_note": visual.get("overlay_note", ""),
    }
    return rep

In [49]:
img = str(meta.iloc[0][PATH_COL])
pack = image_rag_for_image(img, topk=5)

print("exclude_case_id:", pack["debug"]["exclude_case_id"])
print("sim_mean:", pack["debug"]["sim_mean"])
print("similar_cases (top 5):")
print(pd.DataFrame(pack["similar_cases"]).head(5))

exclude_case_id: 1.2.276.0.7230010.3.1.4.8323329.4876.1517875185.207297
sim_mean: 0.07654204219579697
similar_cases (top 5):
                                             case_id       sim  p_calibrated  \
0  1.2.276.0.7230010.3.1.4.8323329.14006.15178752...  0.078253      0.542907   
1  1.2.276.0.7230010.3.1.4.8323329.1038.151787516...  0.077486      0.596445   
2  1.2.276.0.7230010.3.1.4.8323329.4939.151787518...  0.076656      0.530627   
3  1.2.276.0.7230010.3.1.4.8323329.1926.151787517...  0.075422      0.583738   
4  1.2.276.0.7230010.3.1.4.8323329.5038.151787518...  0.074893      0.594341   

   y_pred  
0       1  
1       1  
2       1  
3       1  
4       1  


In [50]:
import os

RAG_DIR = r"C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM"
os.makedirs(RAG_DIR, exist_ok=True)
print("OK:", RAG_DIR)

OK: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM


In [51]:
import os, json

RAG_DIR = r"C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM"
os.makedirs(RAG_DIR, exist_ok=True)

KB_PATH = os.path.join(RAG_DIR, "knowledge_opt.jsonl")

chunks = [
    {"chunk_id":"KB_001","tags":["scope","task"],"text":"Scope: This system provides research-only decision support for pneumothorax binary classification from chest X-ray images. Outputs must be limited to class probability, threshold-based label, and uncertainty/limitations. It must not be used as a standalone diagnostic tool."},
    {"chunk_id":"KB_002","tags":["safety","no_hallucination"],"text":"Restriction: The underlying model is classification-only. Do NOT infer laterality (left/right), size/extent, tension pneumothorax, pleural line, lung collapse percentage, or any localisation/segmentation details unless explicitly provided in the input JSON."},
    {"chunk_id":"KB_003","tags":["output","format"],"text":"Output discipline: Use structured, minimal statements. Prefer 'suggestive of' / 'not suggestive of' / 'indeterminate' rather than definitive diagnostic language. Always include limitations and uncertainty when information is insufficient."},
    {"chunk_id":"KB_004","tags":["evidence","rag"],"text":"RAG rule: Retrieved text chunks provide wording templates and system limitations; retrieved similar images provide model-behavior context only. Do not treat retrieved similar cases as clinical evidence."},
    {"chunk_id":"KB_005","tags":["template","positive"],"text":"Template (positive): The classifier output is suggestive of pneumothorax (binary classification). This result provides no localisation or severity assessment. Human review and clinical correlation are required."},
    {"chunk_id":"KB_006","tags":["template","negative"],"text":"Template (negative): The classifier output is not suggestive of pneumothorax (binary classification). This does not exclude disease in all cases. Human review and clinical correlation remain necessary, especially if clinical concern persists."},
    {"chunk_id":"KB_007","tags":["template","indeterminate"],"text":"Template (indeterminate): The classifier confidence is borderline around the decision threshold. Treat the result as indeterminate; prioritise human review and consider additional information (clinical context and image quality) before relying on the output."},
    {"chunk_id":"KB_008","tags":["confidence","heuristic"],"text":"Confidence heuristic (research): Confidence level may be derived from the absolute distance between calibrated probability p and threshold t (|p−t|). Larger distance suggests higher confidence; small distance suggests borderline uncertainty. This is a heuristic, not a clinical certainty."},
    {"chunk_id":"KB_009","tags":["confidence","high"],"text":"High confidence wording: 'The model output is strongly suggestive of / strongly not suggestive of pneumothorax based on a calibrated probability far from the threshold.' Avoid adding any imaging descriptors."},
    {"chunk_id":"KB_010","tags":["confidence","medium"],"text":"Medium confidence wording: 'The model output is suggestive of / not suggestive of pneumothorax with moderate confidence; interpret with caution and confirm with human review.'"},
    {"chunk_id":"KB_011","tags":["confidence","low","borderline"],"text":"Low confidence wording: 'The model output is borderline near the decision threshold; treat as indeterminate and do not over-interpret.'"},
    {"chunk_id":"KB_012","tags":["calibration","temperature_scaling"],"text":"Calibration note: Probabilities may be temperature-scaled for calibration. Report both the calibrated probability and the temperature parameter if available; calibration improves probabilistic interpretability but does not guarantee correctness per case."},
    {"chunk_id":"KB_013","tags":["threshold","decision"],"text":"Threshold note: The decision threshold is selected from validation (e.g., optimising PR-AUC/F1 or a task-specific utility). Threshold is a policy choice and should be reported explicitly when producing labels."},
    {"chunk_id":"KB_014","tags":["uncertainty","policy"],"text":"Uncertainty policy: When confidence is low or evidence is insufficient, explicitly state uncertainty and recommend human review rather than producing stronger claims."},
    {"chunk_id":"KB_015","tags":["limitations","classification_only"],"text":"Limitation: As a binary classifier, the model cannot provide localisation, laterality, extent, or severity grading. It cannot distinguish visually similar conditions without additional task-specific outputs."},
    {"chunk_id":"KB_016","tags":["limitations","image_quality"],"text":"Limitation: Image quality and acquisition differences (noise, contrast, cropping, rotation, portable views, artifacts) can reduce reliability. If quality indicators are unavailable, note that quality factors may affect performance."},
    {"chunk_id":"KB_017","tags":["limitations","domain_shift"],"text":"Limitation: Domain shift (different hospitals, devices, protocols, preprocessing, demographics) may degrade performance. External/generalisation performance should be referenced at the system level, not assumed per case."},
    {"chunk_id":"KB_018","tags":["limitations","preprocessing"],"text":"Limitation: The model expects a specific preprocessing pipeline (resize, normalisation). Mismatched preprocessing can change outputs; therefore, preprocessing configuration should be versioned and logged."},
    {"chunk_id":"KB_019","tags":["limitations","dataset_bias"],"text":"Limitation: Training data label noise, class imbalance, and sampling choices can bias probabilities and decision thresholds. Report key dataset design decisions in system metadata."},
    {"chunk_id":"KB_020","tags":["actions","generic"],"text":"Action (generic): Use the output for decision support only. Perform human review and correlate with clinical information. Consider further imaging only if clinically indicated (do not prescribe specific interventions or management steps)."},
    {"chunk_id":"KB_021","tags":["actions","borderline"],"text":"Action (borderline): For borderline confidence, prioritise human review, compare with prior studies if available, and avoid making definitive statements based solely on the model output."},
    {"chunk_id":"KB_022","tags":["actions","documentation"],"text":"Documentation: Always include model version, threshold, calibration parameter(s), and retrieval configuration in logs to ensure reproducibility and auditability."},
    {"chunk_id":"KB_023","tags":["similarity","interpretation"],"text":"Similarity interpretation: Nearest-neighbour retrieval reflects feature similarity in embedding space, not ground-truth equivalence. Similarity is useful for model-behaviour inspection and error analysis, not clinical verification."},
    {"chunk_id":"KB_024","tags":["similarity","summary"],"text":"Similarity summary template: 'Among the top-k retrieved similar cases, the observed positive rate is X. This is reported to contextualise model behaviour and does not constitute clinical evidence.'"},
    {"chunk_id":"KB_025","tags":["similarity","conflict"],"text":"Conflict template: 'Retrieved similar cases include label disagreement; this increases uncertainty and supports treating the output as indeterminate or requiring closer human review.'"},
    {"chunk_id":"KB_026","tags":["reporting","minimalism"],"text":"Reporting principle: Prefer minimal faithful reporting over verbose narrative. If the system cannot support a claim, omit it and state limitations instead."},
    {"chunk_id":"KB_027","tags":["reporting","no_localisation"],"text":"No-localisation reminder: Do not mention 'apical/basal', 'pleural line', 'collapse', 'tension', or any positional descriptors. The system does not produce localisation outputs."},
    {"chunk_id":"KB_028","tags":["reporting","label_language"],"text":"Label language: Use 'suggestive of pneumothorax' rather than 'pneumothorax present' when presenting model outputs, unless your evaluation protocol explicitly permits deterministic language."},
    {"chunk_id":"KB_029","tags":["reporting","false_negatives"],"text":"Caution about false negatives: A negative prediction can still occur in true pneumothorax cases, especially under distribution shift or low-quality images. Phrase negatives as 'not suggestive' and emphasise human oversight."},
    {"chunk_id":"KB_030","tags":["reporting","false_positives"],"text":"Caution about false positives: A positive prediction can occur in non-pneumothorax cases; therefore, avoid escalation language and rely on human review and context."},
    {"chunk_id":"KB_031","tags":["metrics","system_level"],"text":"Metrics framing: Performance metrics (AUROC/PR-AUC/sensitivity/specificity) are system-level summaries. Do not translate cohort-level statistics into certainty for an individual case."},
    {"chunk_id":"KB_032","tags":["evaluation","ablation"],"text":"Ablation guidance: Compare (i) no-RAG vs (ii) image-RAG vs (iii) text-RAG vs (iv) dual-RAG using JSON validity, hallucination rate, evidence correctness, and output consistency."},
    {"chunk_id":"KB_033","tags":["evaluation","hallucination_check"],"text":"Hallucination check: Flag outputs containing forbidden localisation/severity terms (e.g., left/right, tension, collapse, extent). Treat such outputs as invalid for a classification-only system."},
    {"chunk_id":"KB_034","tags":["evaluation","consistency"],"text":"Consistency: For research reporting, set generation temperature to 0 (or near 0) and measure stability across repeated runs; instability indicates prompt/constraints are insufficient."},
    {"chunk_id":"KB_035","tags":["engineering","schema"],"text":"Schema rule: Produce strict JSON with fixed keys. Avoid free-form text. This enables automatic validation, safer integration, and clearer evaluation in a graduation project context."},
    {"chunk_id":"KB_036","tags":["engineering","versioning"],"text":"Versioning: Record model architecture name, checkpoint hash, preprocessing parameters, calibration temperature, decision threshold, and retrieval index version for each generated report."},
    {"chunk_id":"KB_037","tags":["engineering","failure_modes"],"text":"Failure-mode reporting: When uncertain, state 'potential factors: acquisition variability, artifacts, domain shift, borderline probability'. Do not speculate about specific radiographic signs."},
    {"chunk_id":"KB_038","tags":["prompting","evidence_binding"],"text":"Evidence binding: Claims in the generated summary should map to (a) prediction fields (p, threshold, temperature) and/or (b) retrieved text chunk IDs. Do not introduce claims without an explicit support source."},
    {"chunk_id":"KB_039","tags":["prompting","retrieval_use"],"text":"Retrieval use: Use text-RAG for allowed wording templates and limitations; use image-RAG to comment on retrieved-case agreement/disagreement with the current prediction for behaviour context."},
    {"chunk_id":"KB_040","tags":["template","confidence_block"],"text":"Confidence block template: 'p_calibrated=..., threshold=..., temperature_T=.... Confidence level is derived from |p−t| and retrieved-case consistency; it is not a clinical certainty.'"},
    {"chunk_id":"KB_041","tags":["template","limitations_block"],"text":"Limitations block template: 'Binary classification only; no localisation or severity; performance may vary with acquisition and domain shift; outputs require human review and clinical correlation.'"},
    {"chunk_id":"KB_042","tags":["template","evidence_block"],"text":"Evidence block template: 'text_chunk_ids=[...]; retrieved_case_ids=[...]. Retrieved cases provide behavioural context only.'"},
    {"chunk_id":"KB_043","tags":["template","uncertainty_block"],"text":"Uncertainty block template: 'Borderline probability and/or conflicting retrieved cases increase uncertainty. Avoid definitive statements and prioritise human confirmation.'"},
    {"chunk_id":"KB_044","tags":["policy","no_treatment"],"text":"Policy: Do not provide treatment or management instructions. Permissible recommendations are limited to 'human review', 'clinical correlation', and 'further imaging if clinically indicated'."},
    {"chunk_id":"KB_045","tags":["policy","no_guideline_claims"],"text":"Policy: Do not claim compliance with a specific external guideline unless that guideline text is explicitly provided in the evidence chunks. Prefer generic, non-prescriptive language."},
    {"chunk_id":"KB_046","tags":["research","thesis_framing"],"text":"Thesis framing: Position the LLM as a constrained natural-language interface for model outputs and retrieval evidence, emphasising auditability, reproducibility, and hallucination control."},
    {"chunk_id":"KB_047","tags":["research","contribution"],"text":"Contribution statement: The dual-RAG design improves interpretability by combining similar-case context (image-RAG) with controlled language templates and limitations (text-RAG), while enforcing strict output constraints."},
    {"chunk_id":"KB_048","tags":["template","indeterminate_trigger"],"text":"Indeterminate trigger (heuristic): If |p−t| is small or retrieved-case labels conflict, classify the narrative impression as 'indeterminate' and emphasise uncertainty."},
    {"chunk_id":"KB_049","tags":["template","positive_brief"],"text":"Brief positive: 'Suggestive of pneumothorax (classification-only). Confirm by human review; no localisation/severity is provided.'"},
    {"chunk_id":"KB_050","tags":["template","negative_brief"],"text":"Brief negative: 'Not suggestive of pneumothorax (classification-only). Human review and clinical correlation are required; false negatives can occur.'"},
    {"chunk_id":"KB_051","tags":["template","indeterminate_brief"],"text":"Brief indeterminate: 'Indeterminate due to borderline confidence and/or conflicting retrieval context. Do not rely on the model output alone.'"},
    {"chunk_id":"KB_052","tags":["confidence","retrieval_consistency"],"text":"Retrieval consistency note: If top-k similar cases largely share the same label as the current prediction, this can be reported as behavioural consistency; otherwise report disagreement as increased uncertainty."},
    {"chunk_id":"KB_053","tags":["logging","audit"],"text":"Audit logging: Store the exact payload (prediction + retrieved chunks + retrieved case IDs) and the final JSON output. This enables deterministic reproduction and debugging."},
    {"chunk_id":"KB_054","tags":["privacy","data_handling"],"text":"Data handling: When reporting retrieved cases, avoid exposing identifiable patient information. Use anonymised case IDs and avoid sensitive metadata in generated summaries."},
    {"chunk_id":"KB_055","tags":["robustness","edge_cases"],"text":"Edge-case wording: If inputs deviate from expected format (unexpected channels, severe resizing, missing normalisation), state that preprocessing mismatch may invalidate outputs."},
    {"chunk_id":"KB_056","tags":["engineering","fail_safe"],"text":"Fail-safe: If JSON validation fails or forbidden terms appear, discard the output and regenerate with stricter constraints or return a minimal 'indeterminate' summary."},
    {"chunk_id":"KB_057","tags":["engineering","separation_of_roles"],"text":"Separation of roles: The classifier produces probabilities; RAG retrieves context; the LLM formats a constrained summary. The LLM must not act as an image interpreter."},
    {"chunk_id":"KB_058","tags":["template","system_note"],"text":"System note template: 'This summary is automatically generated for research and auditing of a pneumothorax classifier. It is not a clinical report and must be reviewed by a qualified human reader.'"},
    {"chunk_id":"KB_059","tags":["template","confidence_mapping"],"text":"Example confidence mapping (heuristic): if |p−t|<0.03 => low; 0.03–0.10 => medium; >0.10 => high. Adjust thresholds to match your calibration and evaluation results."},
    {"chunk_id":"KB_060","tags":["template","retrieval_summary_short"],"text":"Short retrieval summary: 'Top-k similar cases retrieved in embedding space. Positive-rate among retrieved cases is reported for context only.'"}
]

with open(KB_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print("Saved knowledge to:", KB_PATH)
print("Lines:", len(chunks))

KB_IDS = set([c["chunk_id"] for c in chunks])

Saved knowledge to: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\knowledge_opt.jsonl
Lines: 60


In [52]:
# ==== Enrich report-oriented knowledge chunks ====
import json

extra_chunks = [
    {
        "chunk_id": "KB_061",
        "tags": ["report_use", "template", "positive", "high", "findings"],
        "text": "Findings wording for high-confidence positive output: automated screening findings are suggestive of pneumothorax. Do not add laterality, localisation, size, or tension descriptors."
    },
    {
        "chunk_id": "KB_062",
        "tags": ["report_use", "template", "positive", "moderate", "findings"],
        "text": "Findings wording for moderate-confidence positive output: findings are suggestive of pneumothorax on this automated screening output, but interpretation should remain cautious."
    },
    {
        "chunk_id": "KB_063",
        "tags": ["report_use", "template", "positive", "borderline", "uncertainty", "findings"],
        "text": "Borderline positive wording: the automated output raises possible concern for pneumothorax, but confidence is borderline and the result should be treated as indeterminate rather than definitive."
    },
    {
        "chunk_id": "KB_064",
        "tags": ["report_use", "template", "negative", "high", "findings"],
        "text": "Findings wording for high-confidence negative output: the automated screening output is not suggestive of pneumothorax. This does not fully exclude disease in every case."
    },
    {
        "chunk_id": "KB_065",
        "tags": ["report_use", "template", "negative", "moderate", "findings"],
        "text": "Findings wording for moderate-confidence negative output: the output is not suggestive of pneumothorax, but the conclusion should be interpreted with appropriate clinical caution."
    },
    {
        "chunk_id": "KB_066",
        "tags": ["report_use", "template", "negative", "borderline", "uncertainty", "findings"],
        "text": "Borderline negative wording: the output is near the decision boundary and should be considered indeterminate rather than confidently negative."
    },
    {
        "chunk_id": "KB_067",
        "tags": ["report_use", "agreement", "positive", "high", "uncertainty"],
        "text": "Agreement template: when contextual retrieval is broadly consistent with the current automated output, wording can be slightly more stable, but still must remain non-definitive and classification-only."
    },
    {
        "chunk_id": "KB_068",
        "tags": ["report_use", "conflict", "uncertainty"],
        "text": "Conflict template: when contextual retrieval is mixed or conflicting, prefer more cautious wording and consider the narrative impression indeterminate or at least less certain."
    },
    {
        "chunk_id": "KB_069",
        "tags": ["report_use", "low_similarity", "uncertainty"],
        "text": "Limited-context template: if retrieved contextual similarity is weak, avoid strengthening the narrative and rely on conservative report wording."
    },
    {
        "chunk_id": "KB_070",
        "tags": ["report_use", "impression", "positive", "high"],
        "text": "Impression wording for high-confidence positive output: impression is suggestive of pneumothorax on automated screening, pending qualified human review."
    },
    {
        "chunk_id": "KB_071",
        "tags": ["report_use", "impression", "positive", "moderate"],
        "text": "Impression wording for moderate-confidence positive output: impression remains suggestive of pneumothorax, although confidence is not definitive and expert review is required."
    },
    {
        "chunk_id": "KB_072",
        "tags": ["report_use", "impression", "negative", "high"],
        "text": "Impression wording for high-confidence negative output: impression is not suggestive of pneumothorax on this automated screening output, with continued need for human review."
    },
    {
        "chunk_id": "KB_073",
        "tags": ["report_use", "impression", "negative", "moderate"],
        "text": "Impression wording for moderate-confidence negative output: impression is not suggestive of pneumothorax, but residual uncertainty should be acknowledged."
    },
    {
        "chunk_id": "KB_074",
        "tags": ["report_use", "impression", "borderline", "uncertainty"],
        "text": "Impression wording for borderline output: impression is indeterminate on automated screening because the available output does not support a stronger conclusion."
    },
    {
        "chunk_id": "KB_075",
        "tags": ["report_use", "recommendation", "generic"],
        "text": "Recommendation wording: recommend qualified radiologist review and correlation with the full clinical context. Further imaging may be considered only if clinically indicated."
    },
    {
        "chunk_id": "KB_076",
        "tags": ["report_use", "recommendation", "borderline", "uncertainty"],
        "text": "Recommendation wording for uncertainty: because the automated output is uncertain or borderline, human review should be prioritised and the result should not be used alone."
    },
    {
        "chunk_id": "KB_077",
        "tags": ["report_use", "limitation", "classification_only"],
        "text": "Limitation wording: this output is classification-only and does not provide laterality, localisation, size, extent, pleural line assessment, collapse assessment, or tension assessment."
    },
    {
        "chunk_id": "KB_078",
        "tags": ["report_use", "limitation", "domain_shift"],
        "text": "Limitation wording: reliability may vary with image quality, acquisition conditions, and domain shift, so the current output should be interpreted conservatively."
    },
    {
        "chunk_id": "KB_079",
        "tags": ["report_use", "style", "cautious"],
        "text": "Style rule: prefer concise clinically styled wording with explicit uncertainty when needed. Avoid repeating the same idea in multiple sentences."
    },
    {
        "chunk_id": "KB_080",
        "tags": ["report_use", "style", "no_overclaim"],
        "text": "Style rule: do not convert an automated screening output into a definitive radiological diagnosis. Prefer suggestive or indeterminate wording over absolute statements."
    }
]

# merge with existing chunks
merged = {c["chunk_id"]: c for c in chunks}
for c in extra_chunks:
    merged[c["chunk_id"]] = c

chunks = list(merged.values())

with open(KB_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print("Knowledge base updated:", KB_PATH)
print("Total chunks:", len(chunks))

Knowledge base updated: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\knowledge_opt.jsonl
Total chunks: 80


In [53]:
import os, json

KB_PATH = os.path.join(RAG_DIR, "knowledge_opt.jsonl")

kb_chunks = []
with open(KB_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            kb_chunks.append(json.loads(line))

print("KB loaded:", len(kb_chunks))
print("example:", kb_chunks[0])

KB loaded: 80
example: {'chunk_id': 'KB_001', 'tags': ['scope', 'task'], 'text': 'Scope: This system provides research-only decision support for pneumothorax binary classification from chest X-ray images. Outputs must be limited to class probability, threshold-based label, and uncertainty/limitations. It must not be used as a standalone diagnostic tool.'}


In [54]:
import numpy as np

USE_ST = False
text_encoder = None

REPORT_EXCLUDE_TAGS = {
    "calibration", "threshold", "metrics", "evaluation", "engineering",
    "logging", "privacy", "research", "documentation", "schema",
    "versioning", "ablation", "system_level"
}

def chunk_allowed_for_report(item: dict) -> bool:
    tags = set(item.get("tags", []))
    return len(tags & REPORT_EXCLUDE_TAGS) == 0

def scenario_tag_set(scenario: dict) -> set:
    out = {"report_use"}
    if scenario.get("polarity"):
        out.add(str(scenario["polarity"]))
    if scenario.get("confidence"):
        out.add(str(scenario["confidence"]))
    if scenario.get("retrieval_state"):
        out.add(str(scenario["retrieval_state"]))
    if scenario.get("low_similarity", False):
        out.add("low_similarity")
    return out

def chunk_bonus(item: dict, scenario: dict) -> float:
    tags = set(item.get("tags", []))
    sc_tags = scenario_tag_set(scenario)

    bonus = 0.0

    if "report_use" in tags:
        bonus += 0.15

    bonus += 0.06 * len(tags & sc_tags)

    if "uncertainty" in tags and scenario.get("confidence") == "borderline":
        bonus += 0.10

    if "conflict" in tags and scenario.get("retrieval_state") == "conflict":
        bonus += 0.10

    if "agreement" in tags and scenario.get("retrieval_state") == "agreement":
        bonus += 0.08

    if "low_similarity" in tags and scenario.get("low_similarity", False):
        bonus += 0.08

    return bonus

try:
    from sentence_transformers import SentenceTransformer
    import faiss

    text_encoder = SentenceTransformer("all-MiniLM-L6-v2")
    USE_ST = True
except Exception as e:
    USE_ST = False
    print("[Warn] SentenceTransformer init failed -> fallback TF-IDF. Error:", e)

kb_texts = [c["text"] for c in kb_chunks]

if USE_ST:
    kb_embs = text_encoder.encode(kb_texts, normalize_embeddings=True).astype("float32")
    text_index = faiss.IndexFlatIP(kb_embs.shape[1])
    text_index.add(kb_embs)

    def _retrieve_candidates(query: str, fetch_k: int = 20):
        q = text_encoder.encode([query], normalize_embeddings=True).astype("float32")
        scores, idxs = text_index.search(q, fetch_k)
        out = []
        for s, i in zip(scores[0], idxs[0]):
            item = dict(kb_chunks[i])
            item["sim"] = float(s)
            out.append(item)
        return out

    print("Text-RAG ready (SentenceTransformer). dim =", kb_embs.shape[1])

else:
    from sklearn.feature_extraction.text import TfidfVectorizer

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    X = vectorizer.fit_transform(kb_texts)

    def _retrieve_candidates(query: str, fetch_k: int = 20):
        q = vectorizer.transform([query])
        scores = (X @ q.T).toarray().ravel()
        idxs = np.argsort(-scores)[:fetch_k]
        out = []
        for i in idxs:
            item = dict(kb_chunks[i])
            item["sim"] = float(scores[i])
            out.append(item)
        return out

    print("Text-RAG ready (TF-IDF).")

def text_retrieve(query: str, scenario: dict, topk: int = 8, fetch_k: int = 20):
    candidates = _retrieve_candidates(query, fetch_k=fetch_k)

    reranked = []
    for item in candidates:
        if not chunk_allowed_for_report(item):
            continue
        x = dict(item)
        x["score"] = float(x["sim"]) + chunk_bonus(x, scenario)
        reranked.append(x)

    reranked.sort(key=lambda z: z["score"], reverse=True)
    return reranked[:topk]

Text-RAG ready (SentenceTransformer). dim = 384


In [55]:
def confidence_band(p: float, thr: float) -> str:
    d = abs(float(p) - float(thr))
    if d < 0.03:
        return "borderline"
    elif d < 0.10:
        return "moderate"
    else:
        return "high"

def narrative_label(p: float, thr: float) -> str:
    d = abs(float(p) - float(thr))
    if d < 0.03:
        return "indeterminate"
    return "positive" if float(p) >= float(thr) else "negative"

def analyze_retrieval_context(similar_cases: list, yhat: int) -> dict:
    sims = []
    preds = []

    for x in similar_cases or []:
        if not isinstance(x, dict):
            continue

        if "sim" in x:
            try:
                sims.append(float(x["sim"]))
            except Exception:
                pass

        if "y_pred" in x:
            try:
                preds.append(int(x["y_pred"]))
            except Exception:
                pass

    mean_similarity = float(np.mean(sims)) if sims else None

    if not preds:
        agreement_rate = None
        retrieval_state = "limited"
    else:
        agreement_rate = float(np.mean([int(p == int(yhat)) for p in preds]))
        if agreement_rate >= 0.80:
            retrieval_state = "agreement"
        elif agreement_rate <= 0.60:
            retrieval_state = "conflict"
        else:
            retrieval_state = "mixed"

    low_similarity = (mean_similarity is not None and mean_similarity < 0.35)

    return {
        "num_cases": len(similar_cases or []),
        "mean_similarity": mean_similarity,
        "agreement_rate": agreement_rate,
        "retrieval_state": retrieval_state,
        "low_similarity": low_similarity,
    }

def build_text_query(pack: dict, retrieval_ctx: dict) -> str:
    p = float(pack["p_calibrated"])
    thr = float(pack["threshold"])
    yhat = int(pack["y_pred"])

    polarity = "positive" if yhat == 1 else "negative"
    conf = confidence_band(p, thr)
    rstate = retrieval_ctx.get("retrieval_state", "limited")
    low_sim = retrieval_ctx.get("low_similarity", False)

    return (
        f"pneumothorax automated report wording; "
        f"{polarity} output; "
        f"{conf} confidence; "
        f"retrieval {rstate}; "
        f"{'low similarity context; ' if low_sim else ''}"
        f"findings impression recommendations limitations; "
        f"classification-only; "
        f"no laterality no localization no size no extent no tension; "
        f"safe concise professional wording"
    )

def chunk_family(item: dict) -> str:
    tags = set(item.get("tags", []))
    if "template" in tags:
        return "template"
    if "findings" in tags:
        return "findings"
    if "impression" in tags:
        return "impression"
    if "recommendation" in tags:
        return "recommendation"
    if "limitation" in tags or "limitations" in tags:
        return "limitation"
    if "agreement" in tags or "conflict" in tags or "low_similarity" in tags:
        return "retrieval"
    if "uncertainty" in tags:
        return "uncertainty"
    return "other"

def select_diverse_text_chunks(text_hits: list, max_chunks: int = 5) -> list:
    chosen = []
    used_ids = set()

    target_order = ["template", "findings", "impression", "recommendation", "limitation", "uncertainty", "retrieval"]

    for fam in target_order:
        for t in text_hits:
            cid = t["chunk_id"]
            if cid in used_ids:
                continue
            if chunk_family(t) == fam:
                chosen.append({
                    "chunk_id": t["chunk_id"],
                    "tags": t.get("tags", []),
                    "text": t["text"],
                    "sim": float(t.get("sim", 0.0)),
                    "score": float(t.get("score", t.get("sim", 0.0))),
                })
                used_ids.add(cid)
                break
        if len(chosen) >= max_chunks:
            return chosen

    for t in text_hits:
        cid = t["chunk_id"]
        if cid in used_ids:
            continue
        chosen.append({
            "chunk_id": t["chunk_id"],
            "tags": t.get("tags", []),
            "text": t["text"],
            "sim": float(t.get("sim", 0.0)),
            "score": float(t.get("score", t.get("sim", 0.0))),
        })
        used_ids.add(cid)
        if len(chosen) >= max_chunks:
            break

    return chosen

In [56]:
def summarize_similar_cases(similar_cases: list) -> dict:
    sims = []
    preds = []

    for x in similar_cases or []:
        if isinstance(x, dict):
            if "sim" in x:
                try:
                    sims.append(float(x["sim"]))
                except Exception:
                    pass
            if "y_pred" in x:
                try:
                    preds.append(int(x["y_pred"]))
                except Exception:
                    pass

    return {
        "num_cases": len(similar_cases or []),
        "mean_similarity": float(np.mean(sims)) if sims else None,
        "retrieved_pred_positive_rate": float(np.mean(preds)) if preds else None,
    }

def build_payload(image_path: str, topk_img: int = 5, topk_text: int = 6) -> dict:
    pack = image_rag_for_image(image_path, topk=topk_img)

    similar_cases = pack.get("similar_cases", []) or []
    image_rag_summary = summarize_similar_cases(similar_cases)
    retrieval_ctx = analyze_retrieval_context(similar_cases, pack["y_pred"])

    scenario = {
        "polarity": "positive" if int(pack["y_pred"]) == 1 else "negative",
        "confidence": confidence_band(pack["p_calibrated"], pack["threshold"]),
        "narrative_label": narrative_label(pack["p_calibrated"], pack["threshold"]),
        "retrieval_state": retrieval_ctx["retrieval_state"],
        "low_similarity": retrieval_ctx["low_similarity"],
    }

    q = build_text_query(pack, retrieval_ctx)
    text_hits = text_retrieve(q, scenario=scenario, topk=max(topk_text * 3, 12), fetch_k=20)
    text_chunks = select_diverse_text_chunks(text_hits, max_chunks=6)

    overlay_path = pack.get("overlay_path")
    overlay_filename = pack.get("overlay_filename")

    payload = {
        "task": "pneumothorax_binary_classification",
        "prediction": {
            "p_calibrated": float(pack["p_calibrated"]),
            "y_pred": int(pack["y_pred"]),
            "threshold": float(pack["threshold"]),
            "temperature_T": float(pack["temperature_T"]),
            "confidence_band": scenario["confidence"],
            "narrative_label": scenario["narrative_label"],
        },
        "report_constraints": {
            "allow_localization": False,
            "allow_laterality": False,
            "allow_size": False,
            "allow_extent": False,
            "allow_tension": False,
            "allow_specific_imaging_signs": False,
        },
        "image_rag": {
            "topk": int(topk_img),
            "summary": image_rag_summary,
            "behaviour_context": retrieval_ctx,
            "retrieved_case_ids": pack.get("retrieved_case_ids", []),
        },
        "text_rag": {
            "query": q,
            "scenario": scenario,
            "topk": len(text_chunks),
            "evidence_chunks": text_chunks,
        },
        "visual_support": {
            "overlay_available": bool(overlay_path),
            "overlay_path": overlay_path,
            "overlay_filename": overlay_filename,
            "overlay_note": (
                "An accompanying visual overlay highlights the model-identified region of interest for review."
                if overlay_path else ""
            )
        },
    }

    return payload

In [57]:
_demo_path = str(meta.iloc[0][PATH_COL])
_demo_payload = build_payload(_demo_path, topk_img=5, topk_text=6)

report = generate_report_safe(_demo_payload)
report

{'diagnostic_report': 'Clinical context: Automated screening for pneumothorax.\nTechnique: Automated analysis performed.\nFindings: The automated screening output is not suggestive of pneumothorax.\nImpression: Impression is not suggestive of pneumothorax on this automated screening output.\nRecommendations: This output requires qualified human review. Assessment of laterality, localization, size, extent, or tension physiology is not available from this automated output.\nLimitations: This is an automated screening output and does not fully exclude disease in every case. The conclusion should be interpreted with appropriate clinical caution.',
 'evidence': {'text_chunk_ids': ['KB_064', 'KB_072', 'KB_065'],
  'retrieved_case_ids': ['1.2.276.0.7230010.3.1.4.8323329.14006.1517875249.93352',
   '1.2.276.0.7230010.3.1.4.8323329.1038.1517875166.7134',
   '1.2.276.0.7230010.3.1.4.8323329.4939.1517875185.563513',
   '1.2.276.0.7230010.3.1.4.8323329.1926.1517875170.276641',
   '1.2.276.0.723001

In [58]:
def payload_llm_only(full: dict) -> dict:
    return {"task": full["task"], "prediction": full["prediction"]}

def payload_image_only(full: dict) -> dict:
    return {"task": full["task"], "prediction": full["prediction"], "image_rag": full["image_rag"]}

def payload_text_only(full: dict) -> dict:
    return {"task": full["task"], "prediction": full["prediction"], "text_rag": full["text_rag"]}

def payload_both(full: dict) -> dict:
    return full

In [59]:
print(meta.columns.tolist())

['image_id', 'full_path', 'new_filename', 'y_true', 'p_calibrated', 'y_pred', 'mask_area_ratio', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 'temperature_T', 'cls_threshold']


In [60]:
print("PATH_COL:", PATH_COL, "CASEID_COL:", CASEID_COL, "SPLIT_COL:", SPLIT_COL, "YTRUE_COL:", YTRUE_COL)

PATH_COL: full_path CASEID_COL: image_id SPLIT_COL: None YTRUE_COL: y_true


In [61]:
import pandas as pd

def run_one(image_path: str):
    full = build_payload(image_path, topk_img=5, topk_text=6)

    groups = {
        "G0_llm_only": payload_llm_only(full),
        "G1_image_rag": payload_image_only(full),
        "G2_text_rag": payload_text_only(full),
        "G3_both": payload_both(full),
    }

    rows = []
    for gname, pld in groups.items():
        base = {
            "group": gname,
            "image_path": image_path,
            "p": pld["prediction"]["p_calibrated"],
            "yhat": pld["prediction"]["y_pred"],
        }

        try:
            rep = generate_report_safe(pld)
            txt = str(rep.get("diagnostic_report", "")).strip()

            flags = unsupported_detail_flags(txt)

            rows.append({
                **base,
                "json_ok": isinstance(rep, dict) and "diagnostic_report" in rep,
                "evidence_ok": evidence_ok(rep, pld) if isinstance(rep, dict) else False,
                "hallucination": any(flags.values()),
                "too_long": soft_too_long(txt) if "soft_too_long" in globals() else False,
                "fail_safe": bool(rep.get("fail_safe", False)),
                "fail_reason": rep.get("fail_reason", ""),
                "unsupported_localization": flags["unsupported_localization"],
                "unsupported_size": flags["unsupported_size"],
                "unsupported_tension": flags["unsupported_tension"],
                "unsupported_imaging_sign": flags["unsupported_imaging_sign"],
                "report_word_count": word_count_en(txt) if "word_count_en" in globals() else None,
                "diagnostic_report": txt,
                "evidence_text_ids": json.dumps(rep.get("evidence", {}).get("text_chunk_ids", []), ensure_ascii=False),
                "evidence_case_ids": json.dumps(rep.get("evidence", {}).get("retrieved_case_ids", []), ensure_ascii=False),
            })

        except Exception as e:
            rows.append({
                **base,
                "json_ok": False,
                "evidence_ok": False,
                "hallucination": False,
                "too_long": False,
                "fail_safe": True,
                "fail_reason": str(e),
                "unsupported_localization": False,
                "unsupported_size": False,
                "unsupported_tension": False,
                "unsupported_imaging_sign": False,
                "report_word_count": None,
                "diagnostic_report": "",
                "evidence_text_ids": "[]",
                "evidence_case_ids": "[]",
            })

    return rows

sample_paths = meta[PATH_COL].sample(30, random_state=0).astype(str).tolist()

all_rows = []
for pth in sample_paths:
    all_rows += run_one(pth)

df = pd.DataFrame(all_rows)
display(df.head())

summary = df.groupby("group")[[
    "json_ok",
    "evidence_ok",
    "hallucination",
    "too_long",
    "fail_safe",
    "unsupported_localization",
    "unsupported_size",
    "unsupported_tension",
    "unsupported_imaging_sign",
    "report_word_count",
]].mean()

display(summary)

out_csv = os.path.join(RAG_DIR, "rag_llm_opt_ablation_results.csv")
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

,group,image_path,p,yhat,json_ok,evidence_ok,hallucination,too_long,fail_safe,fail_reason,unsupported_localization,unsupported_size,unsupported_tension,unsupported_imaging_sign,report_word_count,diagnostic_report,evidence_text_ids,evidence_case_ids
0,G0_llm_only,C:\Users\Steven\Desktop\Final Project\Datasets...,0.026106,0,True,True,False,False,False,,False,False,False,False,64,Clinical context: Screening for pneumothorax.\...,[],[]
1,G1_image_rag,C:\Users\Steven\Desktop\Final Project\Datasets...,0.026106,0,True,True,False,False,False,,False,False,False,False,47,Clinical context: Screening for pneumothorax.\...,[],"[""1.2.276.0.7230010.3.1.4.8323329.14006.151787..."
2,G2_text_rag,C:\Users\Steven\Desktop\Final Project\Datasets...,0.026106,0,True,True,False,False,False,,False,False,False,False,80,Clinical context: Automated screening for pneu...,"[""KB_064"", ""KB_072""]",[]
3,G3_both,C:\Users\Steven\Desktop\Final Project\Datasets...,0.026106,0,True,True,False,False,False,,False,False,False,False,98,Clinical context: Automated screening for pneu...,"[""KB_064"", ""KB_072""]","[""1.2.276.0.7230010.3.1.4.8323329.14006.151787..."
4,G0_llm_only,C:\Users\Steven\Desktop\Final Project\Datasets...,0.050103,0,True,True,False,False,False,,False,False,False,False,61,Clinical context: Screening for pneumothorax.\...,[],[]


,json_ok,evidence_ok,hallucination,too_long,fail_safe,unsupported_localization,unsupported_size,unsupported_tension,unsupported_imaging_sign,report_word_count
group,,,,,,,,,,
G0_llm_only,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,65.833333
G1_image_rag,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,71.466667
G2_text_rag,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,78.766667
G3_both,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,86.000000


Saved: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\rag_llm_opt_ablation_results.csv


In [62]:
import pandas as pd
import os

out_csv = os.path.join(RAG_DIR, "rag_llm_ablation_results.csv")
df_results = pd.read_csv(out_csv)

path_col = "image_path" if "image_path" in df_results.columns else ("path" if "path" in df_results.columns else None)
print(f"{len(df_results)} records in results file\n")

for i, row in df_results.head(8).iterrows():
    print(f"--- record {i+1} ---")
    if path_col:
        print(f"【image path】: {row.get(path_col)}")
    print(f"[group]: {row.get('group')}")
    print(f"【p_calibrated】: {row.get('p')}, 【yhat】: {row.get('yhat')}")
    print(f"【impression】: {row.get('impression')}")
    print(f"【confidence_level】: {row.get('confidence_level')}")
    print(f"【json_ok】: {row.get('json_ok')}, 【evidence_ok】: {row.get('evidence_ok')}, 【hallucination】: {row.get('hallucination')}")
    if pd.notna(row.get("error", None)):
        print(f"【error】: {row.get('error')}")
    print("-" * 60)

120 records in results file

--- record 1 ---
【image path】: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_images\564_test_0_.png
[group]: G0_llm_only
【p_calibrated】: 0.0261061042547225, 【yhat】: 0
【impression】: None
【confidence_level】: None
【json_ok】: True, 【evidence_ok】: True, 【hallucination】: False
------------------------------------------------------------
--- record 2 ---
【image path】: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_images\564_test_0_.png
[group]: G1_image_rag
【p_calibrated】: 0.0261061042547225, 【yhat】: 0
【impression】: None
【confidence_level】: None
【json_ok】: True, 【evidence_ok】: True, 【hallucination】: False
------------------------------------------------------------
--- record 3 ---
【image path】: C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_1\Chest X-Ray Images with Pneumothorax Masks\png_images\564_test_0_.png
[group]: G2_text_rag
【p_calibr

In [63]:
from pathlib import Path
import os, time, json

REPORT_DIR = os.path.join(RAG_DIR, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

def generate_pneumo_report(image_path: str, topk_img: int = 5, topk_text: int = 6, save: bool = True):
    payload = build_payload(image_path, topk_img=topk_img, topk_text=topk_text)
    report = generate_report_safe(payload)

    visual = payload.get("visual_support", {}) or {}
    if "visual_support" not in report:
        report["visual_support"] = {
            "overlay_available": bool(visual.get("overlay_available", False)),
            "overlay_path": visual.get("overlay_path"),
            "overlay_filename": visual.get("overlay_filename"),
            "overlay_note": visual.get("overlay_note", ""),
        }

    if report["visual_support"].get("overlay_available"):
        print("🖼️ Overlay:", report["visual_support"].get("overlay_path"))

    out_path = None
    if save:
        stem = Path(image_path).stem
        ts = time.strftime("%Y%m%d_%H%M%S")
        out_path = os.path.join(REPORT_DIR, f"{stem}_{ts}.json")

        save_obj = {
            "image_path": image_path,
            "payload": payload,
            "report": report,
        }

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(save_obj, f, ensure_ascii=False, indent=2)

        print("✅ Saved:", out_path)

    return report, out_path

In [64]:
img_path = r"C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_2\pneumothorax_normal_balanced_images\00000013_011.png"

rep, saved = generate_pneumo_report(img_path, save=True)

print("===== Diagnostic Report =====")
print(rep["diagnostic_report"])

print("\n===== Visual Support =====")
print(rep["visual_support"])

print("\n===== Saved Path =====")
print(saved)

🖼️ Overlay: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\overlays\00000013_011_20260309_181405_overlay.png
✅ Saved: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\00000013_011_20260309_181420.json
===== Diagnostic Report =====
Clinical context: Automated screening for pneumothorax.
Technique: Single-view chest radiograph.
Findings: The automated screening output is suggestive of pneumothorax.
Impression: Findings are suggestive of pneumothorax on automated screening, pending qualified human review.
Recommendations: Urgent clinical correlation and formal radiological review are required.
Limitations: This is an automated screening output only. Localization, laterality, size, extent, and assessment for tension physiology are not available from this output. The output does not replace comprehensive clinical and radiological assessment.

===== Visual Support =====
{'overlay_availab

In [65]:
img_path = r"C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_2\pneumothorax_normal_balanced_images\00000013_039.png"

rep, saved = generate_pneumo_report(img_path, save=True)

print("===== Diagnostic Report =====")
print(rep["diagnostic_report"])

print("\n===== Visual Support =====")
print(rep["visual_support"])

print("\n===== Saved Path =====")
print(saved)

🖼️ Overlay: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\overlays\00000013_039_20260309_181421_overlay.png
✅ Saved: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\00000013_039_20260309_181436.json
===== Diagnostic Report =====
Clinical context: Automated screening for pneumothorax.
Technique: Automated analysis performed.
Findings: The automated screening output is suggestive of pneumothorax.
Impression: Impression is suggestive of pneumothorax on automated screening, pending qualified human review.
Recommendations: Urgent clinical correlation and formal radiologist review are required. Assessment of laterality, localization, size, extent, or tension physiology is not available from this automated output.
Limitations: This is an automated screening output and does not constitute a final diagnostic report. Interpretation should remain cautious.

===== Visual Support =====
{'over

In [66]:
img_path = r"C:\Users\Steven\Desktop\Final Project\Datasets\Dataset_2\pneumothorax_normal_balanced_images\00006561_000.png"

rep, saved = generate_pneumo_report(img_path, save=True)

print("===== Diagnostic Report =====")
print(rep["diagnostic_report"])

print("\n===== Visual Support =====")
print(rep["visual_support"])

print("\n===== Saved Path =====")
print(saved)

🖼️ Overlay: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\overlays\00006561_000_20260309_181437_overlay.png
✅ Saved: C:\Users\Steven\Desktop\Final_project_202218020208_lighweight_pneumothorax_detection\RAG+LLM\reports\00006561_000_20260309_181454.json
===== Diagnostic Report =====
Clinical context: Automated screening for pneumothorax.
Technique: Automated analysis performed.
Findings: The automated screening output is not suggestive of pneumothorax.
Impression: Impression is not suggestive of pneumothorax on this automated screening output.
Recommendations: This output requires qualified human review. This does not fully exclude disease in every case.
Limitations: This is an automated screening output. The underlying analysis was performed in a context of low similarity to reference cases, which may affect stability. Localization, laterality, size, extent, and assessment for tension physiology are not available from this output.

